# Lost-words / Zgubione słowa — polski dialogowy pipeline wideo

Automatyczne tworzenie pionowych odcinków survival / dark apocalypse / analog found footage:
**polski scenariusz dialogowy → kilka polskich głosów → kadry 9:16 → napisy PL → ambient → MP4 1080×1920**.

Każda kwestia ma pola line_id, speaker, voice_role, text i visual_prompt. Wypowiedzi są krótkie, rozdzielone naturalnymi pauzami. Obrazowe prompty pozostają po angielsku, ale wszystkie tytuły, dialogi, narracja i napisy są po polsku.

Domyślnie notebook używa scenariusza demonstracyjnego po polsku, Edge TTS i testowych kadrów Pillow. Aby uzyskać obrazy generowane przez API, ustaw IMAGE_PROVIDER = 'openai' i dodaj OPENAI_API_KEY. Bez działającego TTS notebook nie eksportuje filmu udającego gotowy odcinek.

## Otwórz w Google Colab

[Uruchom lost_words_pipeline.ipynb w Google Colab](https://colab.research.google.com/github/simonexex544/lost-words-video-pipeline/blob/main/notebooks/lost_words_pipeline.ipynb)

W Colab wybierz **Środowisko wykonawcze → Uruchom wszystkie** i zaakceptuj montowanie Dysku Google.

## Co robi pipeline

1. Montuje Google Drive w /content/drive/MyDrive/lost_words_project.
2. Konfiguruje temat i lokację odcinka, liczbę kwestii, czas 35–45 s i styl wizualny.
3. Tworzy wyłącznie polskie tytuły, hooki, narrację i dialogi. Każda kwestia ma line_id, speaker, voice_role, text i visual_prompt.
4. Używa kilku polskich głosów Edge TTS, osobnego tempa i pitch dla ról, naturalnych pauz oraz delikatnego filtra radiowego i szumu dla głosów z interkomu.
5. Generuje jeden pionowy kadr na każdą kwestię, z retry i testowym fallbackiem.
6. Montuje ujęcia z lekkim zoom/pan, cichym ambientem, polskimi napisami SRT w bezpiecznej strefie i fontem DejaVu Sans.
7. Eksportuje H.264/AAC, MP4 1080×1920, 30 fps do folderu lost_words_project/output na Google Drive.

## Konfiguracja odcinka

Edytuj komórkę 2:

- EPISODE_TOPIC — np. pociąg przez pustkowie, podziemny schron, opuszczony port, stacja meteorologiczna albo zrujnowane miasto.
- EPISODE_LOCATION — lokacja fabularna.
- NUM_SCENES — liczba kwestii po hooku (4–10); każda kwestia dostaje własny kadr.
- TARGET_SECONDS — 35–45 s.
- PAUSE_BETWEEN_LINES — przerwy między wypowiedziami.
- IMAGE_PROVIDER — fallback do testu albo openai do generowania obrazów.
- LLM_PROVIDER — template albo openai_compatible.

Każdy odcinek powinien zawierać obecność kogoś/czegoś, czego nie powinno tam być, oraz ukryty obiekt, pomieszczenie lub zapis nieobecny w oficjalnym wykazie.

## Głosy i język

Dostępne polskie głosy w domyślnym mapowaniu:

| Rola | Głos Edge TTS | Brzmienie |
|---|---|---|
| calm_dark | pl-PL-MarekNeural | spokojny, obniżony |
| tense_male | pl-PL-MarekNeural | bardziej napięty, niższy pitch |
| calm_female | pl-PL-ZofiaNeural | spokojny żeński |
| radio_distorted | pl-PL-AgnieszkaNeural | pasmo krótkofalówki i szum |
| unknown | pl-PL-MarekNeural | wolniejszy, niższy i przefiltrowany |

Są to polskie głosy wymienione w dokumentacji Microsoft Azure Speech: [Language and voice support](https://learn.microsoft.com/en-us/azure/ai-services/Speech-Service/language-support). Edge TTS nie wymaga klucza API, ale wymaga internetu. Role są symulowane ustawieniami głosu i filtrami audio; to nie jest klonowanie głosu.

## Colab Secrets

W panelu **Secrets** (ikona klucza) dodaj tylko te sekrety, których używasz:

| Sekret | Wymagany? | Przeznaczenie |
|---|---|---|
| OPENAI_API_KEY | opcjonalny | scenariusz przez OpenAI-compatible API; także obrazy przy IMAGE_PROVIDER='openai' |
| OPENAI_BASE_URL | opcjonalny | domyślnie https://api.openai.com/v1 |
| OPENAI_CHAT_MODEL | opcjonalny | domyślnie gpt-4o-mini |
| OPENAI_IMAGE_MODEL | opcjonalny | domyślnie gpt-image-2.5-flare |
| TTS_API_KEY | opcjonalny | placeholder pod własny adapter TTS; Edge TTS go nie potrzebuje |

Po dodaniu sekretu włącz przełącznik udostępniania go temu notebookowi. Nie wpisuj kluczy do komórek kodu ani do repozytorium.

## Pliki i debugowanie

Film: MyDrive/lost_words_project/output/  
Scenariusze JSON: MyDrive/lost_words_project/assets/  
Audio segmentów: MyDrive/lost_words_project/audio/  
Kadry: MyDrive/lost_words_project/frames/

Przy błędzie wklej numer komórki, cały traceback od Traceback (most recent call last): do ostatniej linii oraz komunikaty [INFO], [OSTRZEŻENIE] i [BŁĄD] z bezpośredniego sąsiedztwa błędu. Nie wklejaj sekretów, tokenów ani nagłówków Authorization — zastąp je [UKRYTO].

IMAGE_PROVIDER='fallback' oznacza syntetyczne plansze Pillow do testowania montażu, a nie fotorealistyczne obrazy. Zewnętrzne API mogą być płatne zgodnie z cennikiem dostawcy.


In [ ]:
# KOMÓRKA 1/6 — środowisko, Google Drive, ffmpeg i polskie fonty
from pathlib import Path
import os, sys, json, re, math, time, random, shutil, hashlib, subprocess, textwrap
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    raise RuntimeError(
        "Nie udało się zamontować Google Drive. Uruchom notebook w Google Colab "
        "i zaakceptuj dostęp do Dysku. Szczegóły: " + str(exc)
    ) from exc

PROJECT_ROOT = Path('/content/drive/MyDrive/lost_words_project')
ASSETS_DIR = PROJECT_ROOT / 'assets'
AUDIO_DIR = PROJECT_ROOT / 'audio'
FRAMES_DIR = PROJECT_ROOT / 'frames'
TEMP_DIR = PROJECT_ROOT / 'temp'
OUTPUT_DIR = PROJECT_ROOT / 'output'
for folder in (PROJECT_ROOT, ASSETS_DIR, AUDIO_DIR, FRAMES_DIR, TEMP_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def run_cmd(args, *, check=True, capture_output=True):
    """Uruchamia proces i pokazuje czytelny fragment stderr w razie błędu."""
    try:
        return subprocess.run(
            [str(x) for x in args], check=check, text=True,
            stdout=subprocess.PIPE if capture_output else None,
            stderr=subprocess.PIPE if capture_output else None
        )
    except subprocess.CalledProcessError as exc:
        stderr = (exc.stderr or '')[-5000:]
        print(f"[BŁĄD] Proces zakończył się kodem {exc.returncode}: {args[0]}")
        if stderr:
            print(stderr)
        raise
    except FileNotFoundError as exc:
        raise RuntimeError(f"Nie znaleziono programu {args[0]!r}: {exc}") from exc


# ffmpeg, ffprobe i DejaVu Sans — font z pełnym zestawem polskich znaków.
try:
    missing = [tool for tool in ('ffmpeg', 'ffprobe') if shutil.which(tool) is None]
    if missing or not Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf').exists():
        print('[INFO] Instaluję ffmpeg, ffprobe i DejaVu Sans…')
        run_cmd(['apt-get', 'update', '-qq'], capture_output=False)
        run_cmd([
            'apt-get', 'install', '-y', '-qq',
            'ffmpeg', 'fonts-dejavu-core', 'fontconfig'
        ], capture_output=False)
    if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
        raise RuntimeError('Po instalacji nadal nie znaleziono ffmpeg lub ffprobe.')
    FONT_FILE = Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
    if not FONT_FILE.exists():
        raise RuntimeError(f'Nie znaleziono czcionki DejaVu Sans: {FONT_FILE}')
    print('[OK] Czcionka:', FONT_FILE)
    print('[OK] ffmpeg:', run_cmd(['ffmpeg', '-version']).stdout.splitlines()[0])
except Exception as exc:
    raise RuntimeError(f"Konfiguracja ffmpeg/fontów nie powiodła się: {exc}") from exc

try:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'Pillow', 'edge-tts'])
    import requests
    from PIL import Image, ImageDraw, ImageFilter, ImageEnhance, ImageOps
except Exception as exc:
    raise RuntimeError(f"Instalacja/import pakietów nie powiódł się: {exc}") from exc

try:
    filter_result = run_cmd(['ffmpeg', '-hide_banner', '-filters'])
    filters = (filter_result.stdout or '') + (filter_result.stderr or '')
    if 'subtitles' not in filters:
        raise RuntimeError("Ten build ffmpeg nie zawiera filtra 'subtitles' (libass).")
    print('[OK] Filtr subtitles/libass jest dostępny.')
    print(f'[OK] Folder projektu: {PROJECT_ROOT}')
except Exception as exc:
    raise RuntimeError(f"Nie udało się zweryfikować filtrów ffmpeg: {exc}") from exc


In [ ]:
# KOMÓRKA 2/6 — polska konfiguracja odcinka, role i Colab Secrets

def get_secret(name, default=None):
    """Najpierw Colab Secrets, potem os.environ. Wartości sekretów nigdy nie są wyświetlane."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        value = os.environ.get(name)
    return value if value else default


SERIES_NAME = 'Lost-words / Zgubione słowa'
EPISODE_TOPIC = 'Pociąg przez pustkowie. W składzie pojawia się wagon, którego nie ma w rozkładzie.'
EPISODE_LOCATION = 'Opuszczona linia kolejowa na obrzeżach Łodzi'
STORY_LANGUAGE = 'PL'
NUM_SCENES = 6
TARGET_SECONDS = 40
VIDEO_WIDTH, VIDEO_HEIGHT, FPS = 1080, 1920, 30
VISUAL_STYLE = (
    'gritty analog found footage, dark apocalypse, survival horror, '
    'desaturated colors, 35mm film grain, imperfect handheld documentary realism'
)

PAUSE_BETWEEN_LINES = 0.28
AMBIENT_MIX_GAIN = 0.16
RADIO_STATIC_GAIN = 0.12
ALLOW_SILENT_TEST_AUDIO = False

LLM_PROVIDER = 'openai_compatible'
IMAGE_PROVIDER = 'fallback'
TTS_PROVIDER = 'edge'
FALLBACK_ON_IMAGE_ERROR = True
IMAGE_RETRIES = 3

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
OPENAI_BASE_URL = (get_secret('OPENAI_BASE_URL', 'https://api.openai.com/v1') or '').rstrip('/')
OPENAI_CHAT_MODEL = get_secret('OPENAI_CHAT_MODEL', 'gpt-4o-mini')
OPENAI_IMAGE_MODEL = get_secret('OPENAI_IMAGE_MODEL', 'gpt-image-2.5-flare')
TTS_API_KEY = get_secret('TTS_API_KEY')

if STORY_LANGUAGE != 'PL':
    raise ValueError("Lost-words jest w tej wersji serią wyłącznie polskojęzyczną: STORY_LANGUAGE='PL'.")
if not 4 <= NUM_SCENES <= 10:
    raise ValueError('NUM_SCENES ustaw w zakresie 4–10.')
if not 35 <= TARGET_SECONDS <= 45:
    raise ValueError('TARGET_SECONDS ustaw w zakresie 35–45 sekund.')
if not 0.10 <= PAUSE_BETWEEN_LINES <= 0.60:
    raise ValueError('PAUSE_BETWEEN_LINES ustaw w zakresie 0.10–0.60 s.')
if not 0.05 <= AMBIENT_MIX_GAIN <= 0.30:
    raise ValueError('AMBIENT_MIX_GAIN ustaw w zakresie 0.05–0.30.')

EPISODE_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
EPISODE_ID = re.sub(r'[^a-z0-9]+', '-', f"lost-words-{EPISODE_STAMP}".lower()).strip('-')
FRAME_EPISODE_DIR = FRAMES_DIR / EPISODE_ID
FRAME_EPISODE_DIR.mkdir(parents=True, exist_ok=True)

print(f'[OK] Seria: {SERIES_NAME} | język: polski | kwestii po hooku: {NUM_SCENES}')
print(f'[OK] Temat: {EPISODE_TOPIC}')
print(f'[OK] Lokacja: {EPISODE_LOCATION}')
print(f'[INFO] LLM: {LLM_PROVIDER} | obrazy: {IMAGE_PROVIDER} | TTS: {TTS_PROVIDER}')
print('[INFO] OPENAI_API_KEY:', 'dostępny' if OPENAI_API_KEY else 'brak — scenariusz demonstracyjny')


In [ ]:
# KOMÓRKA 3/6 — scenariusz dialogowy: JSON, role postaci i mroczny motyw

ALLOWED_SPEAKERS = {'narrator', 'ocalaly_1', 'ocalaly_2', 'glos_z_radia', 'nieznany'}
ALLOWED_VOICE_ROLES = {'calm_dark', 'tense_male', 'calm_female', 'radio_distorted', 'unknown'}


def strip_json_fence(raw_text):
    """Usuwa markdownowe bloki JSON i wycina pierwszy kompletny obiekt."""
    if not isinstance(raw_text, str) or not raw_text.strip():
        raise ValueError('Model zwrócił pustą odpowiedź.')
    cleaned = re.sub(r'^\s*\x60{3}(?:json)?\s*', '', raw_text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*\x60{3}\s*$', '', cleaned)
    start, end = cleaned.find('{'), cleaned.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('Nie znaleziono kompletnego obiektu JSON.')
    return cleaned[start:end + 1]


def normalize_dialogue_line(item, expected_id, label):
    if not isinstance(item, dict):
        raise TypeError(f'{label} musi być obiektem JSON.')
    required = ('line_id', 'speaker', 'voice_role', 'text', 'visual_prompt')
    missing = [field for field in required if not isinstance(item.get(field), str) or not item[field].strip()]
    if missing:
        raise ValueError(f'{label}: brak poprawnych pól: {", ".join(missing)}')

    line_id = item['line_id'].strip()
    if not re.fullmatch(r'[A-Za-z0-9_-]{1,20}', line_id):
        raise ValueError(f'{label}: line_id może zawierać wyłącznie litery, cyfry, "_" i "-".')
    speaker = item['speaker'].strip()
    voice_role = item['voice_role'].strip()
    if speaker not in ALLOWED_SPEAKERS:
        raise ValueError(f'{label}: nieznana rola speaker={speaker!r}.')
    if voice_role not in ALLOWED_VOICE_ROLES:
        raise ValueError(f'{label}: nieznana rola głosowa voice_role={voice_role!r}.')

    spoken_text = re.sub(r'\s+', ' ', item['text']).strip()
    if len(spoken_text.split()) > 24:
        raise ValueError(f'{label}: kwestia jest za długa — podziel ją na krótsze wypowiedzi.')
    if not re.search(r'[.!?…]$', spoken_text):
        spoken_text += '.'
    visual_prompt = item['visual_prompt'].strip()
    visual_prompt += (
        ' No visible text, no letters, no logos, no subtitles, no watermark, '
        'no readable signs, no graphic gore.'
    )
    return {
        'line_id': line_id,
        'speaker': speaker,
        'voice_role': voice_role,
        'text': spoken_text,
        'visual_prompt': visual_prompt,
    }


def validate_story(payload):
    if not isinstance(payload, dict):
        raise TypeError('Scenariusz musi być obiektem JSON.')
    if not isinstance(payload.get('title'), str) or not payload['title'].strip():
        raise ValueError('Brak polskiego tytułu.')
    if not isinstance(payload.get('hook'), dict):
        raise ValueError('Pole hook musi być obiektem wypowiedzi.')
    lines = payload.get('lines')
    if not isinstance(lines, list) or len(lines) != NUM_SCENES:
        raise ValueError(f'Wymagane jest dokładnie {NUM_SCENES} kwestii w tablicy lines.')

    hook = normalize_dialogue_line(payload['hook'], 'L00', 'hook')
    hook['line_id'] = 'L00'
    if len(hook['text'].split()) > 9:
        raise ValueError('Hook jest za długi — maksymalnie 9 słów, aby zmieścił się w pierwszych 3 sekundach.')

    normalized = []
    seen_ids = {'L00'}
    for idx, item in enumerate(lines, start=1):
        line = normalize_dialogue_line(item, f'L{idx:02d}', f'lines[{idx - 1}]')
        if line['line_id'] in seen_ids:
            raise ValueError(f"Powtórzony line_id: {line['line_id']}")
        seen_ids.add(line['line_id'])
        normalized.append(line)

    speakers = {line['speaker'] for line in normalized}
    if len(speakers) < 2:
        raise ValueError('Scenariusz musi zawierać rozmowę co najmniej dwóch postaci.')
    return {'title': payload['title'].strip(), 'hook': hook, 'lines': normalized}


def fallback_story():
    """Polski scenariusz demo: ktoś pojawia się w miejscu, którego nie ma w żadnym spisie."""
    prompts = [
        'An abandoned railway carriage at night, one dim red signal light, wet metal floor, handheld documentary framing.',
        'Two exhausted survivors inside a dark passenger carriage, one looking through a rain-streaked window, restrained tension, natural practical light.',
        'A human silhouette standing outside the moving train behind a sealed window, empty wasteland beyond, impossible quiet atmosphere.',
        'A battered analogue train intercom and radio speaker with a tiny blinking indicator, close-up, shallow depth of field, ominous static mood.',
        'A hidden paper passenger ledger inside a loose metal panel, flashlight beam, dusty forbidden compartment, no readable writing.',
        'An empty narrow corridor between train carriages, a third shadow reflected in glass although nobody is there, cinematic analog horror.',
        'A close view of a fogged train window reflecting an empty seat occupied by a vague human silhouette, desaturated 35mm found footage.',
        'A locked service compartment under a train seat, an unlisted metal key and old paper records, flashlight illumination, no readable text.',
        'A deserted rail control room with one monitor glowing in darkness, dusty switches, abandoned infrastructure, documentary realism.',
        'A long train corridor disappearing into darkness, a motionless person at its far end, cold fog and analog film grain.',
    ]
    all_lines = [
        {
            'speaker': 'narrator', 'voice_role': 'calm_dark',
            'text': 'Dziennik. Trzecia siedemnaście. Pociąg jedzie, chociaż od dawna nie ma maszynisty.',
            'visual_prompt': prompts[1],
        },
        {
            'speaker': 'ocalaly_1', 'voice_role': 'tense_male',
            'text': 'Widziałeś kogoś na peronie? Przecież tutaj nie ma już żadnych stacji.',
            'visual_prompt': prompts[2],
        },
        {
            'speaker': 'ocalaly_2', 'voice_role': 'calm_female',
            'text': 'Nie na peronie. Za szybą. Ktoś idzie obok pociągu, dokładnie w naszym tempie.',
            'visual_prompt': prompts[3],
        },
        {
            'speaker': 'glos_z_radia', 'voice_role': 'radio_distorted',
            'text': 'Powtarzam... nie wysyłaliśmy nikogo do tego składu. Nie odpowiadajcie temu człowiekowi.',
            'visual_prompt': prompts[4],
        },
        {
            'speaker': 'ocalaly_1', 'voice_role': 'tense_male',
            'text': 'Znalazłem ukryty rejestr. Jest tu dodatkowy wagon, ale nie ma go w żadnym wykazie.',
            'visual_prompt': prompts[5],
        },
        {
            'speaker': 'nieznany', 'voice_role': 'unknown',
            'text': 'W końcu mnie znaleźliście. Zostawiliście moje nazwisko na samym końcu listy.',
            'visual_prompt': prompts[6],
        },
        {
            'speaker': 'ocalaly_2', 'voice_role': 'calm_female',
            'text': 'To niemożliwe. Ten rejestr jest starszy od nas. A wpis... został dodany przed chwilą.',
            'visual_prompt': prompts[7],
        },
        {
            'speaker': 'narrator', 'voice_role': 'calm_dark',
            'text': 'Zamknąłem notes. Na ostatniej stronie ktoś dopisał godzinę mojego przyjazdu.',
            'visual_prompt': prompts[8],
        },
        {
            'speaker': 'ocalaly_1', 'voice_role': 'tense_male',
            'text': 'Zatrzymaj skład. Słyszysz? Ktoś puka od strony wagonu, którego nie ma.',
            'visual_prompt': prompts[9],
        },
        {
            'speaker': 'nieznany', 'voice_role': 'unknown',
            'text': 'Nie musicie już szukać wyjścia. Właśnie dojechaliście na miejsce.',
            'visual_prompt': prompts[0],
        },
    ]
    for index, line in enumerate(all_lines[:NUM_SCENES], start=1):
        line['line_id'] = f'L{index:02d}'

    payload = {
        'title': 'Wagon, którego nie ma',
        'hook': {
            'line_id': 'L00',
            'speaker': 'glos_z_radia',
            'voice_role': 'radio_distorted',
            'text': 'Nie wchodź tam. Tego wagonu nie ma.',
            'visual_prompt': prompts[0],
        },
        'lines': all_lines[:NUM_SCENES],
    }
    return validate_story(payload)

def generate_story_openai():
    if not OPENAI_API_KEY:
        raise RuntimeError('Brak OPENAI_API_KEY w Colab Secrets.')
    system_prompt = (
        'Jesteś scenarzystą krótkich, mrocznych słuchowisk survivalowych. '
        'Zwróć WYŁĄCZNIE poprawny JSON, bez markdown. '
        'Wszystkie tytuły, kwestie, dialogi i wypowiedzi muszą być wyłącznie po polsku. '
        'Nie używaj angielskich dialogów, romansów, humoru ani szkolnego tonu. '
        'Klimat: upadek cywilizacji, apokalipsa, przetrwanie, utrata łączności, opuszczone miejsca. '
        'W każdym odcinku ktoś lub coś pojawia się tam, gdzie absolutnie nie powinno go być, '
        'a bohaterowie odkrywają ukryty obiekt, pomieszczenie lub zapis nieobecny w oficjalnym wykazie. '
        'Pisz krótkie, naturalne zdania z przecinkami, kropkami, pytajnikami i wielokropkami. '
        'Dialogi mają brzmieć jak prawdziwa rozmowa przez radio/interkom lub między ocalałymi. '
        'Bez graficznej przemocy i makabry. Każdy visual_prompt napisz po angielsku, '
        'w stylu gritty analog found footage / dark apocalypse / survival horror, pionowe 9:16, '
        'i wyraźnie zakazuj tekstu, liter, logotypów, napisów oraz znaków wodnych w obrazie.'
    )
    user_prompt = f"""Przygotuj scenariusz serii "{SERIES_NAME}".
Temat odcinka: {EPISODE_TOPIC}
Lokacja: {EPISODE_LOCATION}
Styl wizualny: {VISUAL_STYLE}
Docelowy czas: około {TARGET_SECONDS} sekund.
Wymagane: jeden hook oraz dokładnie {NUM_SCENES} krótkich kwestii po hooku.

Zwróć JSON dokładnie w strukturze:
{{
  "title": "polski tytuł",
  "hook": {{
    "line_id": "L00",
    "speaker": "glos_z_radia",
    "voice_role": "radio_distorted",
    "text": "polski hook, maksymalnie 9 słów",
    "visual_prompt": "English visual prompt, portrait 9:16, no text in image"
  }},
  "lines": [
    {{
      "line_id": "L01",
      "speaker": "narrator",
      "voice_role": "calm_dark",
      "text": "Krótka naturalna kwestia po polsku.",
      "visual_prompt": "English visual prompt, gritty analog found footage, dark apocalypse, portrait 9:16, no text in image"
    }}
  ]
}}

Dopuszczalne wartości speaker: narrator, ocalaly_1, ocalaly_2, glos_z_radia, nieznany.
Dopuszczalne voice_role: calm_dark, tense_male, calm_female, radio_distorted, unknown.
Wygeneruj dokładnie {NUM_SCENES} elementów w lines, z różnymi speakerami i prawdziwą wymianą zdań.
Hook musi być bardzo mocny i nadawać się na pierwsze 3 sekundy.
Nie dodawaj żadnych pól poza title, hook, lines oraz wymaganymi polami każdej kwestii.
Treść title i każdego text musi być po polsku. visual_prompt zawsze po angielsku.
"""
    response = requests.post(
        f'{OPENAI_BASE_URL}/chat/completions',
        headers={'Authorization': f'Bearer {OPENAI_API_KEY}', 'Content-Type': 'application/json'},
        json={
            'model': OPENAI_CHAT_MODEL,
            'response_format': {'type': 'json_object'},
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            'temperature': 0.85,
        },
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()
    try:
        content = data['choices'][0]['message']['content']
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError('Nieoczekiwany format odpowiedzi API scenariusza.') from exc
    return validate_story(json.loads(strip_json_fence(content)))


try:
    if LLM_PROVIDER == 'openai_compatible' and OPENAI_API_KEY:
        STORY = generate_story_openai()
        print('[OK] Scenariusz dialogowy wygenerowany przez API.')
    elif LLM_PROVIDER in {'template', 'openai_compatible'}:
        STORY = fallback_story()
        print('[INFO] Użyto polskiego scenariusza demonstracyjnego.')
    else:
        raise ValueError("LLM_PROVIDER musi mieć wartość 'openai_compatible' albo 'template'.")
except Exception as exc:
    print(f'[OSTRZEŻENIE] Generowanie scenariusza nie powiodło się: {type(exc).__name__}: {exc}')
    print('[INFO] Przechodzę na polski scenariusz demonstracyjny.')
    STORY = fallback_story()

STORY_PATH = ASSETS_DIR / f'{EPISODE_ID}_story.json'
STORY_PATH.write_text(json.dumps(STORY, ensure_ascii=False, indent=2), encoding='utf-8')

print('\nTYTUŁ:', STORY['title'])
print('HOOK:', STORY['hook']['text'])
for line in STORY['lines']:
    print(f"{line['line_id']} | {line['speaker']} | {line['voice_role']}: {line['text']}")
print(f'[OK] Scenariusz zapisany: {STORY_PATH}')


In [ ]:
# KOMÓRKA 4/6 — polskie głosy Edge TTS, pauzy i filtr krótkofalówki
import asyncio

SEGMENT_SPECS = [STORY['hook']] + STORY['lines']

VOICE_PROFILES = {
    'calm_dark': {'voice': 'pl-PL-MarekNeural', 'rate': '-4%', 'pitch': '-5Hz'},
    'tense_male': {'voice': 'pl-PL-MarekNeural', 'rate': '+1%', 'pitch': '-8Hz'},
    'calm_female': {'voice': 'pl-PL-ZofiaNeural', 'rate': '-3%', 'pitch': '-4Hz'},
    'radio_distorted': {'voice': 'pl-PL-AgnieszkaNeural', 'rate': '-6%', 'pitch': '-10Hz'},
    'unknown': {'voice': 'pl-PL-MarekNeural', 'rate': '-8%', 'pitch': '-12Hz'},
}

SPEAKER_LABELS_PL = {
    'narrator': 'DZIENNIK',
    'ocalaly_1': 'OCALAŁY 1',
    'ocalaly_2': 'OCALAŁA 2',
    'glos_z_radia': 'GŁOS Z RADIA',
    'nieznany': 'NIEZNANY',
}


def probe_duration(path):
    result = run_cmd([
        'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1', str(path)
    ])
    try:
        duration = float(result.stdout.strip())
        if not math.isfinite(duration) or duration <= 0:
            raise ValueError('Niepoprawny czas trwania.')
        return duration
    except Exception as exc:
        raise RuntimeError(f'Nie można odczytać długości {path}: {exc}') from exc


async def create_edge_tts_segment(item, mp3_path):
    profile = VOICE_PROFILES[item['voice_role']]
    try:
        import edge_tts
        communicate = edge_tts.Communicate(
            text=item['text'],
            voice=profile['voice'],
            rate=profile['rate'],
            pitch=profile['pitch'],
            volume='+0%',
        )
        await communicate.save(str(mp3_path))
    except Exception as exc:
        raise RuntimeError(
            f"Edge TTS nie wygenerował {item['line_id']} ({profile['voice']}): "
            f"{type(exc).__name__}: {exc}"
        ) from exc


async def create_external_tts_segment(item, mp3_path):
    """Adapter własnego TTS: zapisz gotowy segment MP3 pod ścieżką mp3_path."""
    if not TTS_API_KEY:
        raise RuntimeError('Własny TTS wymaga sekretu TTS_API_KEY.')
    raise NotImplementedError(
        'Dodaj wywołanie wybranego dostawcy TTS. Adapter powinien obsługiwać różne role głosowe '
        'i zapisać wynik segmentu w mp3_path.'
    )


def convert_segment_to_wav(mp3_path, wav_path, item):
    is_radio = item['voice_role'] in {'radio_distorted', 'unknown'} or item['speaker'] in {'glos_z_radia', 'nieznany'}
    if is_radio:
        filter_complex = (
            '[0:a]aresample=48000,aformat=sample_fmts=fltp:channel_layouts=stereo,'
            'highpass=f=260,lowpass=f=3600,'
            'acompressor=threshold=0.09:ratio=2.4:attack=12:release=120[voice];'
            '[1:a]highpass=f=900,lowpass=f=4200,'
            f'volume={RADIO_STATIC_GAIN:.3f}[static];'
            '[voice][static]amix=inputs=2:duration=first:dropout_transition=0:normalize=0,'
            'alimiter=limit=0.95[out]'
        )
        run_cmd([
            'ffmpeg', '-y', '-i', str(mp3_path),
            '-f', 'lavfi', '-i', 'anoisesrc=color=pink:amplitude=0.010:sample_rate=48000',
            '-filter_complex', filter_complex, '-map', '[out]',
            '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(wav_path)
        ])
    else:
        run_cmd([
            'ffmpeg', '-y', '-i', str(mp3_path),
            '-af', 'aresample=48000,highpass=f=70,lowpass=f=16000,'
                   'acompressor=threshold=0.10:ratio=2.0:attack=18:release=220,'
                   'alimiter=limit=0.95',
            '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(wav_path)
        ])
    if not wav_path.exists() or wav_path.stat().st_size < 4000:
        raise RuntimeError(f'Nie powstał poprawny segment WAV: {wav_path.name}')


def create_pause_wav(path, duration):
    run_cmd([
        'ffmpeg', '-y', '-f', 'lavfi',
        '-i', 'anullsrc=channel_layout=stereo:sample_rate=48000',
        '-t', f'{duration:.3f}', '-ar', '48000', '-ac', '2',
        '-c:a', 'pcm_s16le', str(path)
    ])
    return path


AUDIO_SEGMENT_FILES = []
SEGMENT_DURATIONS = []
SEGMENT_STARTS = []
SEGMENT_PAUSES = []
AUDIO_CONCAT_PARTS = []
cursor = 0.0

try:
    if TTS_PROVIDER not in {'edge', 'external'}:
        raise ValueError("TTS_PROVIDER musi mieć wartość 'edge' albo 'external'.")
    for idx, item in enumerate(SEGMENT_SPECS):
        mp3_path = AUDIO_DIR / f"{EPISODE_ID}_{idx:02d}_{item['line_id']}.mp3"
        wav_path = AUDIO_DIR / f"{EPISODE_ID}_{idx:02d}_{item['line_id']}.wav"

        if TTS_PROVIDER == 'edge':
            await create_edge_tts_segment(item, mp3_path)
        else:
            await create_external_tts_segment(item, mp3_path)

        if not mp3_path.exists() or mp3_path.stat().st_size < 1000:
            raise RuntimeError(f'Plik TTS jest pusty lub niekompletny: {mp3_path.name}')
        convert_segment_to_wav(mp3_path, wav_path, item)

        duration = probe_duration(wav_path)
        SEGMENT_STARTS.append(cursor)
        SEGMENT_DURATIONS.append(duration)
        SEGMENT_PAUSES.append(PAUSE_BETWEEN_LINES if idx < len(SEGMENT_SPECS) - 1 else 0.0)
        AUDIO_SEGMENT_FILES.append(wav_path)
        AUDIO_CONCAT_PARTS.append(wav_path)
        cursor += duration

        if idx < len(SEGMENT_SPECS) - 1:
            pause_path = AUDIO_DIR / f'{EPISODE_ID}_{idx:02d}_pause.wav'
            create_pause_wav(pause_path, PAUSE_BETWEEN_LINES)
            AUDIO_CONCAT_PARTS.append(pause_path)
            cursor += PAUSE_BETWEEN_LINES

        print(
            f"[OK] {item['line_id']} | {SPEAKER_LABELS_PL[item['speaker']]} | "
            f"{VOICE_PROFILES[item['voice_role']]['voice']} | {duration:.2f} s"
        )
except Exception as exc:
    print(f'[BŁĄD TTS] {type(exc).__name__}: {exc}')
    if not ALLOW_SILENT_TEST_AUDIO:
        raise RuntimeError(
            'Nie udało się wygenerować prawdziwej narracji. Nie eksportuję filmu bez dialogów. '
            'Sprawdź połączenie internetowe, nazwę głosu i log błędu powyżej.'
        ) from exc

    print('[OSTRZEŻENIE] Włączono ALLOW_SILENT_TEST_AUDIO — powstanie wyłącznie niemy test montażu.')
    AUDIO_MODE = 'silence-test'
    AUDIO_SEGMENT_FILES, SEGMENT_DURATIONS = [], []
    SEGMENT_STARTS, SEGMENT_PAUSES, AUDIO_CONCAT_PARTS = [], [], []
    cursor = 0.0
    for idx, item in enumerate(SEGMENT_SPECS):
        estimated_duration = max(1.8, len(item['text'].split()) / 2.4)
        silent_path = AUDIO_DIR / f"{EPISODE_ID}_{idx:02d}_{item['line_id']}_silent.wav"
        create_pause_wav(silent_path, estimated_duration)
        SEGMENT_STARTS.append(cursor)
        SEGMENT_DURATIONS.append(estimated_duration)
        pause_duration = PAUSE_BETWEEN_LINES if idx < len(SEGMENT_SPECS) - 1 else 0.0
        SEGMENT_PAUSES.append(pause_duration)
        AUDIO_SEGMENT_FILES.append(silent_path)
        AUDIO_CONCAT_PARTS.append(silent_path)
        cursor += estimated_duration
        if pause_duration:
            pause_path = AUDIO_DIR / f'{EPISODE_ID}_{idx:02d}_silent_pause.wav'
            create_pause_wav(pause_path, pause_duration)
            AUDIO_CONCAT_PARTS.append(pause_path)
            cursor += pause_duration
else:
    AUDIO_MODE = 'edge-tts' if TTS_PROVIDER == 'edge' else 'external-tts'

concat_audio_list = TEMP_DIR / f'{EPISODE_ID}_audio_concat.txt'
concat_audio_list.write_text(
    '\n'.join(f"file '{path.as_posix()}'" for path in AUDIO_CONCAT_PARTS),
    encoding='utf-8'
)
VOICE_TRACK = AUDIO_DIR / f'{EPISODE_ID}_dialogue_with_pauses.wav'
run_cmd([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(concat_audio_list),
    '-ar', '48000', '-ac', '2', '-c:a', 'pcm_s16le', str(VOICE_TRACK)
])
VOICE_TRACK_SECONDS = probe_duration(VOICE_TRACK)
print(f'[OK] Dialog z pauzami: {VOICE_TRACK}')
print(f'[INFO] Czas ścieżki dialogowej: {VOICE_TRACK_SECONDS:.2f} s')


In [ ]:
# KOMÓRKA 5/6 — pionowe kadry dla każdej kwestii, retry i fallback testowy
import base64
from io import BytesIO


def normalize_image_9x16(image, output_path):
    """Centralny crop do dokładnego 9:16, RGB i 1080×1920."""
    image = image.convert('RGB')
    target_ratio = 9 / 16
    width, height = image.size
    ratio = width / height
    if ratio > target_ratio:
        new_width = int(height * target_ratio)
        left = (width - new_width) // 2
        image = image.crop((left, 0, left + new_width, height))
    elif ratio < target_ratio:
        new_height = int(width / target_ratio)
        top = (height - new_height) // 2
        image = image.crop((0, top, width, top + new_height))
    image = image.resize((VIDEO_WIDTH, VIDEO_HEIGHT), Image.Resampling.LANCZOS)
    image.save(output_path, format='PNG', optimize=True)
    return Path(output_path)


def create_test_frame(item, index, output_path):
    """Syntetyczny kadr testowy — wyraźnie odróżniony od obrazu z generatora AI."""
    seed = int(hashlib.sha256((item['line_id'] + item['visual_prompt']).encode('utf-8')).hexdigest()[:8], 16)
    rng = random.Random(seed)
    width, height = 540, 960
    base = Image.new('RGB', (width, height))
    pixels = base.load()
    for y in range(height):
        gradient = y / max(1, height - 1)
        shade = int(20 + 34 * (1 - gradient))
        for x in range(width):
            noise = rng.randint(-7, 7)
            pixels[x, y] = (max(0, shade + noise - 5), max(0, shade + noise - 3), max(0, shade + noise))
    fog = Image.new('RGBA', (width, height), (0, 0, 0, 0))
    fog_draw = ImageDraw.Draw(fog)
    for _ in range(18):
        x, y = rng.randint(-100, width + 80), rng.randint(100, height - 100)
        rx, ry = rng.randint(50, 150), rng.randint(18, 65)
        fog_draw.ellipse((x-rx, y-ry, x+rx, y+ry), fill=(175, 180, 180, rng.randint(8, 24)))
    base = Image.alpha_composite(base.convert('RGBA'), fog).convert('RGB')
    draw = ImageDraw.Draw(base)
    horizon = int(height * (0.56 + 0.03 * (index % 3)))
    for building in range(7):
        bw = rng.randint(38, 95)
        bx = building * 85 - 15 + rng.randint(-14, 14)
        bh = rng.randint(100, 270)
        shade = 12 + rng.randint(0, 10)
        draw.rectangle((bx, horizon-bh, bx+bw, horizon+45), fill=(shade, shade+1, shade+3))
        for wy in range(horizon-bh+20, horizon, 34):
            for wx in range(bx+10, bx+bw-8, 25):
                if rng.random() > 0.78:
                    draw.rectangle((wx, wy, wx+7, wy+12), fill=(78, 74, 57))
    if index % 2 == 0:
        sx = width // 2 + rng.randint(-45, 45)
        draw.ellipse((sx-14, horizon-200, sx+14, horizon-172), fill=(5, 6, 8))
        draw.ellipse((sx-28, horizon-178, sx+28, horizon-70), fill=(7, 8, 10))
    antenna_x = width // 2 + rng.randint(-75, 75)
    draw.line((antenna_x, horizon-260, antenna_x, horizon-410), fill=(8, 9, 10), width=4)
    draw.line((antenna_x-45, horizon-390, antenna_x+45, horizon-390), fill=(8, 9, 10), width=3)
    glow = Image.new('RGBA', (width, height), (0, 0, 0, 0))
    glow_draw = ImageDraw.Draw(glow)
    light_x, light_y = rng.randint(90, width-90), rng.randint(230, 560)
    glow_draw.ellipse((light_x-42, light_y-42, light_x+42, light_y+42), fill=(194, 157, 89, 25))
    glow_draw.ellipse((light_x-8, light_y-8, light_x+8, light_y+8), fill=(210, 181, 125, 110))
    base = Image.alpha_composite(base.convert('RGBA'), glow).convert('RGB')
    base = ImageEnhance.Contrast(base).enhance(1.18).filter(ImageFilter.GaussianBlur(0.35))
    grain = Image.effect_noise((width, height), 34).convert('RGB')
    base = Image.blend(base, grain, 0.10)
    return normalize_image_9x16(base, output_path)


def generate_openai_image(item, output_path):
    if not OPENAI_API_KEY:
        raise RuntimeError('IMAGE_PROVIDER=openai wymaga OPENAI_API_KEY.')
    prompt = (
        f"{item['visual_prompt']}. Style: {VISUAL_STYLE}. "
        "Vertical portrait 9:16 composition, cinematic still frame, realistic lighting, "
        "survival horror atmosphere. No text, no letters, no logos, no subtitles, no watermark."
    )
    response = requests.post(
        f'{OPENAI_BASE_URL}/images/generations',
        headers={'Authorization': f'Bearer {OPENAI_API_KEY}', 'Content-Type': 'application/json'},
        json={
            'model': OPENAI_IMAGE_MODEL,
            'prompt': prompt,
            'size': '1008x1792',
            'quality': 'medium',
            'output_format': 'png',
        },
        timeout=180,
    )
    response.raise_for_status()
    payload = response.json()
    entries = payload.get('data') or []
    if not entries:
        raise ValueError('API obrazów zwróciło pustą tablicę data.')
    image_item = entries[0]
    if image_item.get('b64_json'):
        image_bytes = base64.b64decode(image_item['b64_json'])
    elif image_item.get('url'):
        image_response = requests.get(image_item['url'], timeout=120)
        image_response.raise_for_status()
        image_bytes = image_response.content
    else:
        raise ValueError('Brak b64_json i url w odpowiedzi API obrazów.')
    with Image.open(BytesIO(image_bytes)) as image:
        normalize_image_9x16(image, output_path)


def generate_frame_with_retry(item, index):
    output_path = FRAME_EPISODE_DIR / f"{index:02d}_{item['line_id']}.png"
    last_error = None
    for attempt in range(1, IMAGE_RETRIES + 1):
        try:
            if IMAGE_PROVIDER == 'fallback':
                create_test_frame(item, index, output_path)
            elif IMAGE_PROVIDER == 'openai':
                generate_openai_image(item, output_path)
            else:
                raise ValueError("IMAGE_PROVIDER musi mieć wartość 'fallback' albo 'openai'.")
            if not output_path.exists() or output_path.stat().st_size < 1000:
                raise RuntimeError('Wygenerowany kadr jest pusty lub zbyt mały.')
            with Image.open(output_path) as check_image:
                if check_image.size != (VIDEO_WIDTH, VIDEO_HEIGHT):
                    raise ValueError(f'Nieprawidłowy rozmiar kadru: {check_image.size}')
            return output_path
        except Exception as exc:
            last_error = exc
            print(
                f"[OSTRZEŻENIE] Kadr {item['line_id']}, "
                f"próba {attempt}/{IMAGE_RETRIES}: {type(exc).__name__}: {exc}"
            )
            time.sleep(min(2 ** (attempt - 1), 8))

    if FALLBACK_ON_IMAGE_ERROR and IMAGE_PROVIDER != 'fallback':
        print(f"[INFO] Używam syntetycznego kadru testowego dla {item['line_id']}.")
        return create_test_frame(item, index, output_path)
    raise RuntimeError(f"Nie udało się wygenerować kadru {item['line_id']}: {last_error}")


FRAME_PATHS = []
for idx, item in enumerate(SEGMENT_SPECS, start=1):
    frame = generate_frame_with_retry(item, idx)
    FRAME_PATHS.append(frame)
    print(f"[OK] {item['line_id']} | {item['speaker']} | {frame.name}")

frame_manifest = {
    'episode_id': EPISODE_ID,
    'provider': IMAGE_PROVIDER,
    'frames': [str(path) for path in FRAME_PATHS],
}
(FRAME_EPISODE_DIR / 'manifest.json').write_text(
    json.dumps(frame_manifest, ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'[OK] Kadry: {len(FRAME_PATHS)} | katalog: {FRAME_EPISODE_DIR}')


In [ ]:
# KOMÓRKA 6/6 — napisy PL, płynny ruch kamery, ambient i eksport H.264
from IPython.display import display, Video


def srt_timestamp(seconds):
    millis = max(0, int(round(seconds * 1000)))
    hours, remainder = divmod(millis, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, millis = divmod(remainder, 1000)
    return f'{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}'


def wrap_caption(text, width=30):
    return '\n'.join(textwrap.wrap(
        text.strip(), width=width,
        break_long_words=False, break_on_hyphens=False
    ))


subtitle_entries = []
for item, start, duration in zip(SEGMENT_SPECS, SEGMENT_STARTS, SEGMENT_DURATIONS):
    label = SPEAKER_LABELS_PL.get(item['speaker'], 'GŁOS')
    caption = item['text'] if item['speaker'] == 'narrator' else f'{label}: {item["text"]}'
    subtitle_entries.append((start, start + duration, caption))

srt_path = TEMP_DIR / f'{EPISODE_ID}_pl.srt'
srt_lines = []
for number, (start, end, caption) in enumerate(subtitle_entries, start=1):
    srt_lines.extend([
        str(number),
        f'{srt_timestamp(start)} --> {srt_timestamp(end)}',
        wrap_caption(caption, width=30),
        '',
    ])
srt_path.write_text('\n'.join(srt_lines), encoding='utf-8')

clip_durations = [
    duration + pause
    for duration, pause in zip(SEGMENT_DURATIONS, SEGMENT_PAUSES)
]
if len(FRAME_PATHS) != len(clip_durations):
    raise RuntimeError(
        f'Liczba kadrów ({len(FRAME_PATHS)}) nie pasuje do liczby kwestii ({len(clip_durations)}).'
    )

clip_paths = []
for idx, (frame_path, duration) in enumerate(zip(FRAME_PATHS, clip_durations), start=1):
    clip_path = TEMP_DIR / f'{EPISODE_ID}_clip_{idx:02d}.mp4'
    frame_count = max(1, int(math.ceil(duration * FPS)))
    pan_x = "iw/2-(iw/zoom/2)" if idx % 2 else "iw/2-(iw/zoom/2)+on*0.015"
    pan_y = "ih/2-(ih/zoom/2)"
    zoom_filter = (
        "zoompan=z='min(1.0+on/1900\\,1.075)':"
        f"x='{pan_x}':y='{pan_y}':"
        f"d=1:s={VIDEO_WIDTH}x{VIDEO_HEIGHT}:fps={FPS},setsar=1,format=yuv420p"
    )
    try:
        run_cmd([
            'ffmpeg', '-y', '-loop', '1', '-framerate', str(FPS), '-i', str(frame_path),
            '-vf', zoom_filter, '-frames:v', str(frame_count), '-an',
            '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20',
            '-pix_fmt', 'yuv420p', '-r', str(FPS), str(clip_path)
        ])
    except Exception as exc:
        raise RuntimeError(f'Nie udało się wyrenderować ujęcia {idx}: {exc}') from exc
    clip_paths.append(clip_path)
    print(f'[OK] Ujęcie {idx}/{len(FRAME_PATHS)}: {duration:.2f} s')

concat_video_list = TEMP_DIR / f'{EPISODE_ID}_video_concat.txt'
concat_video_list.write_text(
    '\n'.join(f"file '{path.as_posix()}'" for path in clip_paths),
    encoding='utf-8'
)
VIDEO_ONLY = TEMP_DIR / f'{EPISODE_ID}_video_only.mp4'
run_cmd([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(concat_video_list),
    '-an', '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '19',
    '-pix_fmt', 'yuv420p', '-r', str(FPS), str(VIDEO_ONLY)
])

OUTPUT_VIDEO = OUTPUT_DIR / f'{EPISODE_ID}.mp4'
subtitle_filename = srt_path.as_posix().replace('\\', '\\\\').replace(':', '\\:')
subtitle_filter = (
    f"subtitles=filename='{subtitle_filename}':"
    "force_style='FontName=DejaVu Sans,FontSize=46,PrimaryColour=&H00FFFFFF,"
    "OutlineColour=&HCC000000,BorderStyle=1,Outline=3,Shadow=1,Alignment=2,"
    "MarginV=350,MarginL=90,MarginR=90,WrapStyle=2'"
)
ambient_source = (
    f"anoisesrc=color=pink:amplitude=0.012:sample_rate=48000:"
    f"duration={VOICE_TRACK_SECONDS + 0.5:.3f}"
)
audio_filter = (
    '[1:a]highpass=f=70,lowpass=f=16000,'
    'acompressor=threshold=0.10:ratio=2.0:attack=18:release=220,'
    'alimiter=limit=0.95[voice];'
    '[2:a]highpass=f=55,lowpass=f=620,volume='
    f'{AMBIENT_MIX_GAIN:.3f}[ambient];'
    '[voice][ambient]amix=inputs=2:duration=first:dropout_transition=0:'
    'weights=\'1 1\':normalize=0,alimiter=limit=0.95[aout]'
)
try:
    run_cmd([
        'ffmpeg', '-y',
        '-i', str(VIDEO_ONLY),
        '-i', str(VOICE_TRACK),
        '-f', 'lavfi', '-i', ambient_source,
        '-vf', subtitle_filter,
        '-filter_complex', audio_filter,
        '-map', '0:v:0', '-map', '[aout]',
        '-shortest',
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '20',
        '-pix_fmt', 'yuv420p', '-r', str(FPS),
        '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart',
        str(OUTPUT_VIDEO)
    ])
except Exception as exc:
    raise RuntimeError(
        'Eksport MP4 nie powiódł się. Sprawdź libass, czcionkę DejaVu Sans i filtry audio ffmpeg. '
        f'Szczegóły: {type(exc).__name__}: {exc}'
    ) from exc

if not OUTPUT_VIDEO.exists() or OUTPUT_VIDEO.stat().st_size < 100_000:
    raise RuntimeError(f'Plik wyjściowy nie istnieje lub jest podejrzanie mały: {OUTPUT_VIDEO}')

probe = run_cmd([
    'ffprobe', '-v', 'error', '-show_streams', '-show_format',
    '-of', 'json', str(OUTPUT_VIDEO)
])
probe_data = json.loads(probe.stdout)
video_stream = next(
    (stream for stream in probe_data.get('streams', []) if stream.get('codec_type') == 'video'),
    None
)
audio_stream = next(
    (stream for stream in probe_data.get('streams', []) if stream.get('codec_type') == 'audio'),
    None
)
if (
    not video_stream
    or int(video_stream.get('width', 0)) != VIDEO_WIDTH
    or int(video_stream.get('height', 0)) != VIDEO_HEIGHT
):
    raise RuntimeError('Niepoprawna rozdzielczość — oczekiwano 1080×1920.')
if video_stream.get('codec_name') != 'h264':
    raise RuntimeError('Niepoprawny kodek wideo — oczekiwano H.264.')
if not audio_stream or audio_stream.get('codec_name') != 'aac':
    raise RuntimeError('Brak poprawnej ścieżki audio AAC.')
final_duration = float(probe_data.get('format', {}).get('duration', 0))
if not 30 <= final_duration <= 50:
    print(f'[OSTRZEŻENIE] Film ma {final_duration:.2f} s; sprawdź tempo dialogów i długość kwestii.')

print('\n' + '=' * 62)
print('[GOTOWE] Lost-words / Zgubione słowa')
print('Tytuł:', STORY['title'])
print('Plik:', OUTPUT_VIDEO)
print(f'Parametry: {VIDEO_WIDTH}×{VIDEO_HEIGHT}, {FPS} fps, H.264, AAC, {final_duration:.2f} s')
print(f'Kwestie: {len(SEGMENT_SPECS)} | język: polski | audio: {AUDIO_MODE}')
print('Głosy:', ', '.join(sorted({VOICE_PROFILES[item['voice_role']]['voice'] for item in SEGMENT_SPECS})))
print('Filtr radiowy: włączony dla głosu z radia i nieznanego.')
print('Ambient: cichy szum otoczenia pod dialogiem.')
print('Kadry:', IMAGE_PROVIDER, '(fallback oznacza plansze testowe Pillow).')
print('=' * 62)
display(Video(str(OUTPUT_VIDEO), embed=True, width=360))
